# 09 缺失值与特征工程消融实验

本轮增强在 validation-based 流程下系统比较缺失值处理和基础特征工程策略。所有策略、模型和阈值选择都在 valid 上完成，official test 只用于最终评估。

本 notebook 不做 GridSearch、不做 SHAP、不做公开 baseline 对比，也不重做风险分层。

## 1. 为什么做消融实验

Day 2 已经发现 Scania APS 存在明显缺失值问题，并且 pos/neg 缺失模式差异较大。消融实验的目标是验证：

- 高缺失字段是否应该删除；
- 缺失指示变量是否有帮助；
- 缺失模式本身是否携带预测信号；
- XGBoost 原生缺失处理是否优于 median imputation；
- 低方差、高相关、L1 选择是否值得保留。

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from scania_aps.config import get_config
from scania_aps.data.load_data import load_train_test_with_target
from scania_aps.data.split_data import split_train_valid

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

cfg = get_config(PROJECT_ROOT / "config" / "config.yaml")
cfg.feature_ablation["enable_strategies"]

['median_all',
 'drop_80_missing_median',
 'drop_50_missing_median',
 'median_with_indicator',
 'xgb_native_missing',
 'low_variance_filter',
 'high_correlation_filter',
 'l1_feature_selection',
 'missing_indicator_only']

## 2. train_inner / valid 划分

本轮继续沿用 validation-based 流程：从 official train 中分出 train_inner 和 valid，official test 不参与策略或阈值选择。

In [2]:
train_df, test_df = load_train_test_with_target(cfg)
train_inner_df, valid_df = split_train_valid(train_df, cfg)

pd.DataFrame([
    {"dataset": "official_train", "rows": len(train_df), "pos": int(train_df["target"].sum()), "pos_rate": train_df["target"].mean()},
    {"dataset": "train_inner", "rows": len(train_inner_df), "pos": int(train_inner_df["target"].sum()), "pos_rate": train_inner_df["target"].mean()},
    {"dataset": "valid", "rows": len(valid_df), "pos": int(valid_df["target"].sum()), "pos_rate": valid_df["target"].mean()},
    {"dataset": "official_test", "rows": len(test_df), "pos": int(test_df["target"].sum()), "pos_rate": test_df["target"].mean()},
])

,dataset,rows,pos,pos_rate
0,official_train,60000,1000,0.016667
1,train_inner,48000,800,0.016667
2,valid,12000,200,0.016667
3,official_test,16000,375,0.023438


## 3. 读取消融实验结果

如果以下文件不存在，请先从项目根目录运行：

```powershell
python scripts/07_feature_ablation_experiments.py
```

In [3]:
valid_best = pd.read_csv(cfg.metrics_dir / "feature_ablation_valid_best_summary.csv")
test_results = pd.read_csv(cfg.metrics_dir / "feature_ablation_final_test_results.csv")
metadata = pd.read_csv(cfg.tables_dir / "feature_ablation_strategy_metadata.csv")
missing_signal = pd.read_csv(cfg.tables_dir / "missing_indicator_signal_summary.csv")

valid_best.shape, test_results.shape, metadata.shape, missing_signal.shape

((16, 22), (16, 19), (16, 13), (2, 14))

## 4. valid 上各策略表现

valid 是策略和阈值选择依据。下表按 valid total cost 排序。

In [4]:
valid_best[[
    "selection_rank", "model_name", "strategy", "best_threshold",
    "precision", "recall", "f2", "fp", "fn", "total_cost",
    "average_precision", "n_features", "n_dropped_features"
]].head(16)

,selection_rank,model_name,strategy,best_threshold,precision,recall,f2,fp,fn,total_cost,average_precision,n_features,n_dropped_features
0,1,xgboost_scale_pos_weight,drop_50_missing_median,0.14,0.340278,0.980,0.712209,380,4,5800,0.867949,162,8
1,2,xgboost_scale_pos_weight,median_with_indicator,0.10,0.305901,0.985,0.682133,447,3,5970,0.865209,339,0
2,3,xgboost_scale_pos_weight,low_variance_filter,0.12,0.330523,0.980,0.703518,397,4,5970,0.859301,169,1
3,4,xgboost_scale_pos_weight,high_correlation_filter,0.13,0.327759,0.980,0.701001,402,4,6020,0.859182,145,25
4,5,xgboost_scale_pos_weight,xgb_native_missing,0.18,0.381139,0.970,0.741024,315,6,6150,0.880775,170,0
5,6,xgboost_scale_pos_weight,drop_80_missing_median,0.14,0.331633,0.975,0.702450,393,5,6430,0.867203,168,2
6,7,xgboost_scale_pos_weight,median_all,0.16,0.359259,0.970,0.723881,346,6,6460,0.867239,170,0
7,8,logistic_regression_balanced,high_correlation_filter,0.69,0.433962,0.920,0.751634,240,16,10400,0.728234,145,25
8,9,logistic_regression_balanced,median_with_indicator,0.51,0.343808,0.930,0.693512,355,14,10550,0.722236,339,0
9,10,logistic_regression_balanced,low_variance_filter,0.38,0.326316,0.930,0.678832,384,14,10840,0.722866,169,1


## 5. official test 最终评估

每个模型和策略都使用其 valid 上选出的 best threshold，在 official test 上评估一次。注意：下表只能用于观察泛化表现，不能反过来用 test 选择策略。

In [ ]:
test_results[[
    "model_name", "strategy", "threshold", "precision", "recall",
    "f2", "average_precision", "fp", "fn", "total_cost",
    "n_features", "n_dropped_features"
]].head(16)

,model_name,strategy,threshold,precision,recall,f2,average_precision,fp,fn,total_cost,n_features,n_dropped_features
0,xgboost_scale_pos_weight,median_with_indicator,0.10,0.396963,0.976000,0.755574,0.910751,556,9,10060,339,0
1,xgboost_scale_pos_weight,drop_80_missing_median,0.14,0.436975,0.970667,0.780111,0.910463,469,11,10190,168,2
2,xgboost_scale_pos_weight,low_variance_filter,0.12,0.420809,0.970667,0.769556,0.907502,501,11,10510,169,1
3,xgboost_scale_pos_weight,drop_50_missing_median,0.14,0.429586,0.968000,0.773987,0.906715,482,12,10820,162,8
4,xgboost_scale_pos_weight,median_all,0.16,0.455919,0.965333,0.789015,0.908623,432,13,10820,170,0
5,xgboost_scale_pos_weight,xgb_native_missing,0.18,0.494505,0.960000,0.807899,0.909038,368,15,11180,170,0
6,xgboost_scale_pos_weight,high_correlation_filter,0.13,0.423032,0.960000,0.765632,0.905827,491,15,12410,145,25
7,logistic_regression_balanced,drop_50_missing_median,0.24,0.356426,0.946667,0.711138,0.792798,641,20,16410,162,8
8,logistic_regression_balanced,low_variance_filter,0.38,0.441509,0.936000,0.764706,0.797103,444,24,16440,169,1
9,logistic_regression_balanced,l1_feature_selection,0.35,0.423402,0.936000,0.753542,0.796899,478,24,16780,163,7


## 6. 缺失模式是否有信息

`missing_indicator_only` 只使用每个字段是否缺失的 0/1 指示变量，不使用原始数值。如果它的 PR-AUC 明显高于正类基准率，说明缺失模式本身携带预测信号。

In [6]:
missing_signal

,model_name,strategy,valid_best_threshold,valid_average_precision,valid_pos_rate,valid_total_cost,test_average_precision,test_pos_rate,test_recall,test_f2,test_fp,test_fn,test_total_cost,signal_note
0,xgboost_scale_pos_weight,missing_indicator_only,0.78,0.461624,0.016667,13330,0.471252,0.023438,0.898667,0.625464,857,38,27570,AP 高于正类基准率，说明缺失模式本身存在一定信号
1,logistic_regression_balanced,missing_indicator_only,0.64,0.368189,0.016667,15070,0.435308,0.023438,0.885333,0.583890,1011,43,31610,AP 高于正类基准率，说明缺失模式本身存在一定信号


## 7. drop_50 / drop_80 / median_all 对比

高缺失字段不能只凭缺失率删除。这里比较保留全部特征、删除 >=80% 缺失字段、删除 >=50% 缺失字段。

In [7]:
compare_missing_drop = test_results[
    test_results["strategy"].isin(["median_all", "drop_80_missing_median", "drop_50_missing_median"])
][["model_name", "strategy", "threshold", "recall", "f2", "fp", "fn", "total_cost", "n_features", "n_dropped_features"]]
compare_missing_drop.sort_values(["model_name", "total_cost"])

,model_name,strategy,threshold,recall,f2,fp,fn,total_cost,n_features,n_dropped_features
7,logistic_regression_balanced,drop_50_missing_median,0.24,0.946667,0.711138,641,20,16410,162,8
10,logistic_regression_balanced,drop_80_missing_median,0.31,0.938667,0.738255,532,23,16820,168,2
11,logistic_regression_balanced,median_all,0.36,0.933333,0.755287,467,25,17170,170,0
1,xgboost_scale_pos_weight,drop_80_missing_median,0.14,0.970667,0.780111,469,11,10190,168,2
3,xgboost_scale_pos_weight,drop_50_missing_median,0.14,0.968000,0.773987,482,12,10820,162,8
4,xgboost_scale_pos_weight,median_all,0.16,0.965333,0.789015,432,13,10820,170,0


## 8. xgb_native_missing 是否有优势

XGBoost 原生支持 NaN。本实验比较它与 median imputation 的成本和召回表现。

In [8]:
test_results[test_results["strategy"].isin(["xgb_native_missing", "median_all", "drop_80_missing_median", "median_with_indicator"])]\
    [["model_name", "strategy", "threshold", "precision", "recall", "f2", "fp", "fn", "total_cost"]]\
    .sort_values("total_cost")

,model_name,strategy,threshold,precision,recall,f2,fp,fn,total_cost
0,xgboost_scale_pos_weight,median_with_indicator,0.10,0.396963,0.976000,0.755574,556,9,10060
1,xgboost_scale_pos_weight,drop_80_missing_median,0.14,0.436975,0.970667,0.780111,469,11,10190
4,xgboost_scale_pos_weight,median_all,0.16,0.455919,0.965333,0.789015,432,13,10820
5,xgboost_scale_pos_weight,xgb_native_missing,0.18,0.494505,0.960000,0.807899,368,15,11180
10,logistic_regression_balanced,drop_80_missing_median,0.31,0.398190,0.938667,0.738255,532,23,16820
11,logistic_regression_balanced,median_all,0.36,0.428397,0.933333,0.755287,467,25,17170
12,logistic_regression_balanced,median_with_indicator,0.51,0.469471,0.922667,0.773357,391,29,18410


## 9. 低方差 / 高相关 / L1 是否有价值

这些策略更偏向基础特征工程和模型简化。如果 total cost 没有改善，不应强行包装成有效。

In [9]:
test_results[test_results["strategy"].isin(["low_variance_filter", "high_correlation_filter", "l1_feature_selection", "median_all"])]\
    [["model_name", "strategy", "threshold", "recall", "f2", "fp", "fn", "total_cost", "n_features", "n_dropped_features"]]\
    .sort_values(["model_name", "total_cost"])

,model_name,strategy,threshold,recall,f2,fp,fn,total_cost,n_features,n_dropped_features
8,logistic_regression_balanced,low_variance_filter,0.38,0.936000,0.764706,444,24,16440,169,1
9,logistic_regression_balanced,l1_feature_selection,0.35,0.936000,0.753542,478,24,16780,163,7
11,logistic_regression_balanced,median_all,0.36,0.933333,0.755287,467,25,17170,170,0
13,logistic_regression_balanced,high_correlation_filter,0.69,0.890667,0.791844,275,41,23250,145,25
2,xgboost_scale_pos_weight,low_variance_filter,0.12,0.970667,0.769556,501,11,10510,169,1
4,xgboost_scale_pos_weight,median_all,0.16,0.965333,0.789015,432,13,10820,170,0
6,xgboost_scale_pos_weight,high_correlation_filter,0.13,0.960000,0.765632,491,15,12410,145,25


## 10. 结论和下一步

- valid 上最低成本组合是 XGBoost + `drop_50_missing_median` + threshold 0.14。
- official test 观察中，`median_with_indicator` 的 total cost 最低，但这不能作为 test 反选策略的依据。
- `missing_indicator_only` 的 AP 明显高于正类基准率，说明缺失模式本身确实有预测信号；但单独使用缺失指示变量的成本仍较高，不能替代原始数值特征。
- `xgb_native_missing` 的 F2 较高、FP 较少，但 FN 增加导致 total cost 不占优。
- 低方差过滤是轻量可保留策略；高相关过滤和 L1 在当前实验中没有成为主线方案。

下一步若继续增强，建议基于 validation 流程做轻量级 XGBoost 调参或模型解释性分析。